In [1]:
import pandas as pd
import numpy as np
import re
import string
import os
import warnings
warnings.filterwarnings('ignore')

### NLP

### pip install vaderSentiment

In [2]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer  # Fixed: use nltk vader (matches streamlit_app.py)


In [3]:
# ML

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

### Download required NLTK data(run once)

In [5]:
nltk.download('stopwords',quiet=True)
nltk.download('punkt',quiet=True)
nltk.download('wordnet',quiet=True)
nltk.download('averaged_perceptron_tagger',quiet=True)
nltk.download('punkt_tab', quiet=True)

True

## Step 1

In [6]:
def load_dataset(filepath ='fake_job_postings.csv'):

    print("=" * 60)
    print("STEP1 : Loading dataset...")
    print("=" * 60)



In [7]:
df= pd.read_csv('fake_job_postings.csv')

print(f" Total records : {len(df):,}")
print(f" Columns : {df.shape[1]}")
print(f" fake Posting : {df['fraudulent'].sum():,} ({(df['fraudulent']==1).mean()*100 :.1f}%)")
print(f" Real Posting : {(df['fraudulent']==0).sum():,} ({(df['fraudulent']==0).mean()*100:.1f}%)")
print(f" Missing Values : \n{df.isnull().sum()[df.isnull().sum()>0]}\n")

 Total records : 17,880
 Columns : 18
 fake Posting : 866 (4.8%)
 Real Posting : 17,014 (95.2%)
 Missing Values : 
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
dtype: int64



In [8]:
print(df.columns)

Index(['job_id', 'title', 'location', 'department', 'salary_range',
       'company_profile', 'description', 'requirements', 'benefits',
       'telecommuting', 'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent'],
      dtype='object')


### BASIC CLEANING

## STEP 2

In [9]:
# STEP 2: Basic Cleaning
print("=" * 60)
print("STEP 2: Basic Cleaning...")
print("=" * 60)
df = df.copy()

### Text Columns - Fill NaN with empty string

In [10]:
text_cols=['title','location','department','salary_range','company_profile',
           'description','requirements','benefits','employment_type',
           'required_experience','required_education','industry','function']


In [11]:
for col in text_cols:
    if col in df.columns:
        df[col]=df[col].fillna('').astype(str).str.strip()

### Boolean columns - fill with 0

In [12]:
bool_cols = ['telecommuting', 'has_company_logo', 'has_questions']

for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)  # Fixed: assign back


### Combine all text fields into one master text column

In [13]:
df['full_text'] = (
    df['title']+' '+
    df['company_profile']+' '+
    df['description']+' '+
    df['requirements']+' '+
    df['benefits']
)

print(f" Cleaning done. Shape: {df.shape}\n")

 Cleaning done. Shape: (17880, 19)



## STEP 3 TEXT PREPROCESSING

In [14]:
def preprocess_text(text):
    if not isinstance(text , str) or text.strip() == '':
        return ''

    # Lowercase
    text = text.lower()

    #Remove HTML Tags
    text=re.sub(r'<[^>]=+>',' ',text)

    #Remove URLs
    text = re.sub(r'http\s+|www\s+',' ',text)

    #Remove email addresses
    text = re.sub(r'\s+@\s+',' Email ', text)

    #Remove Phone numbers(Various Formats)
    text = re.sub(r'(\+91[\-\s]?)[6-9]\d{9}', ' PHONE ', text)
    text = re.sub(r'\b\d{3}[\-\s]\d{3}[\-\s]\d{4}\b',' PHONE ',text)

    #Keep Currency Symbols for analysis, then remove
    text = re.sub(r'[₹$€£]',' CURRENCY ',text)

    #Remove Punctuation & digits
    text = re.sub(r'[^a-z\s]',' ',text)

    #Tokenize
    tokens = word_tokenize(text)

    #Remove Stopwords (Keep some important ones for fraud detection)
    fraud_keywords = {'free', 'urgent', 'immediately', 'guaranteed','no experience','work from home','earn','daily','weekly','payment'}
    stop_words = set(stopwords.words('english')) - fraud_keywords
    tokens = [t for t in tokens if t not in stop_words and len(t)>2]

    #Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return' '.join(tokens)

def apply_text_preprocessing(df):
    print("STEP3: Text Preprocessing(may take 2-3 minutes)...")
    df = df.copy()
    df['cleaned_text'] = df['full_text'].apply(preprocess_text)
    df['cleaned_description']=df['description'].apply(preprocess_text)
    print(f" Text preprocessing done.\n")
    return df



In [15]:
# Cell 1 — rerun Step 3 with the fix
df = apply_text_preprocessing(df)

STEP3: Text Preprocessing(may take 2-3 minutes)...
 Text preprocessing done.



In [16]:
# Cell 2 — check df is not None
print(type(df))        # should print <class 'pandas.core.frame.DataFrame'>
print(df.shape)        # should print something like (17880, 25)

<class 'pandas.core.frame.DataFrame'>
(17880, 21)


## STEP 4 FEATURE ENGINEERING - TEXT FEATURES

In [17]:
# India specific fraud vocabulary

In [18]:
INDIA_FRAUD_WORDS =[
    # Existing patterns (keep all)
    'urgent hiring', 'immediate joining', 'no experience required',
    'earn from home', 'daily payment', 'weekly payment',
    'registration fee', 'processing fee', 'security deposit',
    'guaranteed job', '100% placement',
    'crypto payment', 'earn in dollars',
    'amazon work from home', 'flipkart hiring',
    'zomato delivery partner', 'swiggy partner',
    'online captcha job', 'ad posting job',
    'copy paste job', 'data typing job',
    'like and earn', 'refer and earn',
    'task based earning', 'per task payment',

    # ALL Indian job portal fraud patterns (new) ↓

    # LinkedIn
    'spotted your linkedin',
    'we reviewed your linkedin profile',
    'selected based on linkedin',
    'linkedin profile shortlisted',
    'congratulations linkedin',
    'apply via linkedin only',

    # Naukri
    'found your naukri profile',
    'your naukri resume selected',
    'naukri profile shortlisted',
    'we saw your naukri profile',
    'update your naukri profile',
    'naukri premium job',
    'naukri recruiter contacted',

    # Indeed India
    'indeed job offer',
    'selected on indeed',
    'your indeed profile',
    'indeed easy apply selected',
    'indeed recruiter',
    'apply on indeed only',

    # Internshala
    'internshala internship offer',
    'selected on internshala',
    'internshala recruiter',
    'your internshala profile',
    'internshala premium',
    'internshala direct hire',

    # Shine
    'shine.com profile selected',
    'found your shine profile',
    'shine job offer',
    'shine recruiter',

    # Fresherworld / Freshersworld
    'freshersworld profile',
    'fresherworld job offer',
    'selected on freshersworld',

    # Monster India
    'monster india profile',
    'found your monster profile',
    'monster job offer',
    'monster recruiter',

    # Apna app (very popular in 2025-26)
    'apna app job',
    'apna recruiter',
    'selected on apna',
    'apna interview',
    'apna app selected',

    # IIMJobs / Hirist (for experienced)
    'iimjobs profile selected',
    'hirist profile shortlisted',

    # Wellfound / AngelList
    'wellfound job offer',
    'angellist recruiter',
    'startup job wellfound',

    # Common cross-portal fraud patterns
    'your profile has been selected',
    'we found your profile on',
    'our team reviewed your profile',
    'congratulations your profile',
    'your resume has been shortlisted',
    'we came across your profile',
    'you have been selected for',
    'direct joining offer',
    'offer letter ready',
    'joining date confirmed',
]


URGENCY_WORDS = [
    'urgent','immediately','asap','right away','today','now','instant','quick','fast','hurry','limited','last chance','closing','deadline','expire',
    'last few seats','only 3 seats left','batch starting monday','batch filling fast','interview today only','offer expires tonight',
    'grab this opportunity','dont miss out'
]



SUSPICIOUS_CONTACT = [
    # Messaging apps
    'whatsapp',
    'telegram',
    'instagram',
    'facebook',
    'snapchat',

    # NEW — LinkedIn specific red flags ↓
    'linkedin.com/in',        # fake personal linkedin profiles used as contact
    'connect on linkedin',    # asking to connect instead of official apply
    'message on linkedin',    # DM based hiring = red flag
    'linkedin easy apply',    # mass spam apply abuse

    # Free email domains
    'gmail.com',
    'yahoo.com',
    'yahoo.in',
    'outlook.com',
    'rediffmail.com',
    'protonmail.com',
    'tempmail.com',
]


OVERPROMISE_WORDS = [
    "guaranteed", "100% profit", "risk free",
    "double your money", "earn instantly",
    "no loss", "sure return"
]


INDIAN_JOB_PORTALS = [
    'linkedin', 'naukri', 'indeed', 'internshala',
    'shine', 'monster', 'freshersworld', 'apna',
    'iimjobs', 'hirist', 'wellfound', 'angellist',
    'timesjobs', 'quikr jobs', 'olx jobs'
]

print("All word lists defined successfully!")

All word lists defined successfully!


In [19]:
# Define portal fraud detector function

In [20]:
def has_portal_fraud_pattern(text):
    text_lower = str(text).lower()
    selection_words = [
    'selected','shortlisted','chosen','picked',
    'offer','joining','congratulations','hiring'
    ]
    for portal in INDIAN_JOB_PORTALS:
        if portal in text_lower:
            for word in selection_words:
                if word in text_lower:
                    return 1
    return 0


def has_linkedin_fraud_pattern(text):
    text_lower = str(text).lower()
    linkedin_fraud_patterns = [
    'we came across your linkedin','we reviewed your linkedin profile','your linkedin profile caught','found your profile on linkedin',
    'spotted you on linkedin','you have been selected based on your linkedin','linkedin profile has been shortlisted','congratulations linkedin',
    'apply via linkedin only','urgent linkedin requirement'
    ]
    return int(any(p in text_lower for p in linkedin_fraud_patterns))

print("Portal fraud funtions defined!")

Portal fraud funtions defined!


In [21]:
## Define extract text features function

In [22]:
def extract_text_features(df):
    print("STEP 4: Extracting text features...")
    df = df.copy()
    sia = SentimentIntensityAnalyzer()

    #Basic text statistics
    df['desc_word_count']        =df['description'].apply(lambda x: len(str(x).split()))
    df['desc_char_count']        =df['description'].apply(lambda x: len(str(x)))
    df['desc_sentence_count']    =df['description'].apply(lambda x: len(re.split(r'[.!?]', str(x))))
    df['title_word_count']       =df['title'].apply(lambda x: len(str(x).split()))
    df['avg_word_length']        =df['description'].apply(lambda x: round(np.mean([len(w) for w in str(x).split()]), 4) if str(x).split() else 0)

    #Punctuation abuse
    df['exclamation_count']       = df['full_text'].apply(lambda x: str(x).count('!'))
    df['question_mark_count']     = df['full_text'].apply(lambda x: str(x).count('?'))
    df['caps_ratio']              = df['full_text'].apply(lambda x: round(sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1), 4))

#India fraud keyword counts
    def count_fraud_words(text):
        text_lower = str(text).lower()
        return sum(1 for word in INDIA_FRAUD_WORDS if word in text_lower)
        
    def count_urgency_words(text):
        text_lower = str(text).lower()
        return sum(1 for word in URGENCY_WORDS if word in text_lower)

    def count_overpromise_words(text):
        text_lower = str(text).lower()
        return sum(1 for word in OVERPROMISE_WORDS if word in text_lower)
        
        
    df['india_fraud_score']          =df['full_text'].apply(count_fraud_words)
    df['india_urgency_score']        =df['full_text'].apply(count_urgency_words)
    df['india_overpromise_score']    =df['full_text'].apply(count_overpromise_words) 
        
      #contact red flags
    df['has_whatsapp_contact']       =df['full_text'].apply(lambda x:  int('whatsapp' in str(x).lower()))
    df['has_suspicious_email']       =df['full_text'].apply(lambda x:  int(any(d in str(x).lower() for d in SUSPICIOUS_CONTACT)))
    df['has_phone_in_desc']          =df['description'].apply(lambda x:int(bool(re.search(r' (\+91[\-\s]?)?[6-9]\d{9}', str(x)))))

    # Fee and document red flags
    def has_fee_language(text):
        text_lower = str(text).lower()
        fee_patterns = ['registration fee', 'processing fee', 'security deposit','training fee','pay to apply','refundable deposit']
        return int(any(p in text_lower for p in fee_patterns))


    def requests_personal_docs(text):
        text_lower = str(text).lower()
        doc_patterns = [
            'aadhar','aadhaar','pan card','passport copy','bank account','bank details','upi id','gpay number','phonepay number','paytm number',
            'crypto wallet','selfie with id','video kyc','electricity bill','driving licence','voter id',
        ]
        return int(any(p in text_lower for p in doc_patterns))
        
    df['has_fee_language']         =df['full_text'].apply(has_fee_language)
    df['requests_personal_docs']   =df['full_text'].apply(requests_personal_docs)
        
     # Portal fraud patterns
    df['has_linkedin_fraud']       =df['full_text'].apply(has_linkedin_fraud_pattern)
    df['has_portal_fraud_pattern'] =df['full_text'].apply(has_portal_fraud_pattern)

    # VADER Sentiment
    def get_sentiment(text):
        scores = sia.polarity_scores(str(text)[:512])
        return scores['compound'], scores['pos'], scores['neg'], scores['neu']
       
    sentiment_values           = df['description'].apply(get_sentiment)
       
    df['sentiment_compound']   = sentiment_values.apply(lambda x: round(x[0], 4))
    df['sentiment_positive']   = sentiment_values.apply(lambda x: round(x[1], 4))
    df['sentiment_negative']   = sentiment_values.apply(lambda x: round(x[2], 4))
    df['sentiment_neutral']    = sentiment_values.apply(lambda x: round(x[3], 4))

#Repetition score
    def repetition_score(text):
        words = str(text).lower().split()
        if len(words) < 10:
            return 0.0
        return round(1 - len(set(words)) / len(words), 4)
    df['repetition_score']  = df['description'].apply(repetition_score)
    print(f" Text features extracted sucessfully!")
    print(f" New columns added: {df.shape[1]}")
    return df

print("extract_text_features function defined!")

extract_text_features function defined!


In [23]:
df = extract_text_features(df)
print(df[['india_fraud_score', 'india_urgency_score','has_portal_fraud_pattern','fraudulent']].head(10))

STEP 4: Extracting text features...
 Text features extracted sucessfully!
 New columns added: 44
   india_fraud_score  india_urgency_score  has_portal_fraud_pattern  \
0                  0                    3                         0   
1                  0                    4                         0   
2                  0                    0                         0   
3                  0                    3                         0   
4                  0                    3                         0   
5                  0                    0                         0   
6                  0                    3                         0   
7                  0                    2                         0   
8                  0                    1                         0   
9                  0                    1                         0   

   fraudulent  
0           0  
1           0  
2           0  
3           0  
4           0  
5           0  
6        

## Step 5 structural features

In [24]:
# Cell 2 — run Step 5 structural features (you skipped this!)
def extract_structural_features(df):
    print("STEP 5: Extracting structural features...")
    df = df.copy()

    # Salary range analysis
    def parse_salary(salary_str):
        if not salary_str or salary_str.strip() == '':
            return 0, 0, 1
        nums = re.findall(r'\d+', str(salary_str).replace(',', ''))
        if len(nums) >= 2:
            return int(nums[0]), int(nums[1]), 0
        elif len(nums) == 1:
            return int(nums[0]), int(nums[0]), 0
        return 0, 0, 1

    salary_parsed                = df['salary_range'].apply(parse_salary)
    df['salary_min']             = salary_parsed.apply(lambda x: x[0])
    df['salary_max']             = salary_parsed.apply(lambda x: x[1])
    df['salary_missing']         = salary_parsed.apply(lambda x: x[2])
    df['salary_range_width']     = df['salary_max'] - df['salary_min']

    df['salary_too_high_for_fresher'] = (
        (df['salary_max'] > 250000) &
        (df['required_experience'].str.lower().str.contains(
            'no experience|fresher|entry|0 year', na=False))
    ).astype(int)

    df['salary_unrealistic']     = (df['salary_max'] > 800000).astype(int)

    # Company profile
    df['company_profile_missing'] = (df['company_profile'].str.strip() == '').astype(int)
    df['company_profile_length']  = df['company_profile'].apply(lambda x: len(str(x)))
    df['has_company_logo']        = df['has_company_logo'].astype(int)
    df['has_questions']           = df['has_questions'].astype(int)

    # Job title red flags
    def is_vague_title(title):
        vague_patterns = ['data entry', 'home based', 'work from home',
                          'part time', 'earn money', 'online job',
                          'simple job', 'easy job', 'typing work',
                          'form filling', 'copy paste']
        title_lower = str(title).lower()
        return int(any(p in title_lower for p in vague_patterns))

    df['vague_job_title']         = df['title'].apply(is_vague_title)

    # Requirements and benefits
    df['requirements_missing']    = (df['requirements'].str.strip() == '').astype(int)
    df['requirements_length']     = df['requirements'].apply(lambda x: len(str(x)))
    df['benefits_missing']        = (df['benefits'].str.strip() == '').astype(int)

    # Employment type encoding
    emp_type_map = {
        'full-time': 0, 'part-time': 1, 'contract': 2,
        'temporary': 3, 'other': 4, '': 5
    }
    df['employment_type_encoded'] = df['employment_type'].str.lower().map(
        lambda x: emp_type_map.get(x, 5)
    )

    # Experience encoding
    def encode_experience(exp):
        exp = str(exp).lower()
        if 'not applicable' in exp or exp.strip() == '':
            return -1
        elif 'entry' in exp or 'internship' in exp:
            return 0
        elif 'associate' in exp or '1-2' in exp:
            return 1
        elif 'mid-senior' in exp or '2-5' in exp:
            return 2
        elif 'director' in exp or 'executive' in exp:
            return 3
        return 0

    df['experience_encoded']      = df['required_experience'].apply(encode_experience)

    # Location analysis
    def is_generic_location(location):
        generic = ['anywhere', 'worldwide', 'remote', 'work from home',
                   'india', 'pan india', 'all india', 'multiple cities']
        loc_lower = str(location).lower()
        return int(any(g in loc_lower for g in generic))

    df['generic_location']        = df['location'].apply(is_generic_location)

    # Missing fields score
    df['missing_fields_score']    = (
        df['company_profile_missing'] +
        df['requirements_missing'] +
        df['benefits_missing'] +
        df['salary_missing']
    )

    # Urgency and overpromise (correct names matching ENGINEERED_FEATURES)
    def count_urgency_words(text):
        text_lower = str(text).lower()
        return sum(1 for word in URGENCY_WORDS if word in text_lower)

    def count_overpromise_words(text):
        text_lower = str(text).lower()
        return sum(1 for word in OVERPROMISE_WORDS if word in text_lower)

    df['urgency_score']           = df['full_text'].apply(count_urgency_words)
    df['overpromise_score']       = df['full_text'].apply(count_overpromise_words)

    print("  Structural features extracted successfully!")
    print(f"  Total columns now: {df.shape[1]}")
    return df

print("extract_structural_features defined!")

extract_structural_features defined!


In [25]:
# Cell 3 — call Step 5
df = extract_structural_features(df)

STEP 5: Extracting structural features...
  Structural features extracted successfully!
  Total columns now: 62


## STEP 6: TF - IDF VECTORIZATION

In [26]:
def build_tfidf_features(df, max_features=3000):
    print(f" STEP 6: Building TF -IDF features (top {max_features} terms)...")

    vectorizer = TfidfVectorizer(
        max_features = max_features,
        ngram_range = (1,2),
        min_df = 3,
        max_df = 0.85,
        sublinear_tf = True,
        strip_accents = 'unicode',
        analyzer = 'word'
    )

    tfidf_matrix = vectorizer.fit_transform(df['cleaned_text'].fillna(''))
    print(f" TF-IDF matrix shape: {tfidf_matrix.shape}\n")
    return vectorizer, tfidf_matrix

print("build_tfidf_features defined!")

build_tfidf_features defined!


In [27]:
# Cell 2 — CALL the function (this is what was missing!)
vectorizer, tfidf_matrix = build_tfidf_features(df, max_features=3000)

 STEP 6: Building TF -IDF features (top 3000 terms)...
 TF-IDF matrix shape: (17880, 3000)



In [28]:
# Cell 3 — verify output
print("Vectorizer vocab size:", len(vectorizer.vocabulary_))
print("TF-IDF matrix shape  :", tfidf_matrix.shape)

Vectorizer vocab size: 3000
TF-IDF matrix shape  : (17880, 3000)


## STEP 7: ASSEMBLE FINAL FEATURE MATRIX

In [29]:
## Define Engineered features list

In [30]:
ENGINEERED_FEATURES = [
    #Text statistics
    'desc_word_count', 'desc_char_count', 'desc_sentence_count','title_word_count','avg_word_length',
    #Punctuation abuse
    'exclamation_count','question_mark_count','caps_ratio',
    #India fraud signals
    'india_fraud_score', 'urgency_score', 'overpromise_score',
    #Contact red flags
    'has_whatsapp_contact','has_suspicious_email', 'has_phone_in_desc',
    #Fee and document flags
    'has_fee_language','requests_personal_docs',
    #Portsl fraud
    'has_linkedin_fraud', 'has_portal_fraud_pattern',
    #Sentiment
    'sentiment_compound', 'sentiment_positive',
    'sentiment_negative', 'sentiment_neutral',
    #Repetition
    'repetition_score',
    # Salary (added in Step 5)
    'salary_min', 'salary_max','salary_missing','salary_range_width','salary_too_high_for_fresher','salary_unrealistic',
    #Company info (added in step 5)
    'company_profile_missing','company_profile_length','has_company_logo','has_questions',
    #Job title (added in Step 5)
    'vague_job_title',
    #Requirements and benefits (added in Step 5)
    'requirements_missing', 'requirements_length','benefits_missing',
    #Encodings (added in Step 5)
    'employment_type_encoded','experience_encoded',
    #Location (added in Step 5)
    'generic_location',
    #Summary score (added in Step 5)
    'missing_fields_score',
    #Telecommuting
    'telecommuting',
]

print(f" ENGINEERED_FEATURES defined - total features: {len(ENGINEERED_FEATURES)}")
    

 ENGINEERED_FEATURES defined - total features: 42


In [31]:
def assemble_feature_matrix(df, tfidf_matrix):
    print("STEP 7: Assembling final feature matrix...")
    import scipy.sparse as sp

    #Engineered features as dense array
    X_engineered = df[ENGINEERED_FEATURES].fillna(0).values.astype(np.float32)

    #Scale engineered features
    scaler = StandardScaler()
    X_engineered_scaled = scaler.fit_transform(X_engineered)

    # Combine: sparse TF -IDF + dense engineered deatures
    X_final = sp.hstack([
        tfidf_matrix,
        sp.csr_matrix(X_engineered_scaled)
    ])
    y = df['fraudulent'].values
    print(f" Final feature matrix shape : {X_final.shape}")
    print(f" Target distribution        : {np.bincount(y)}\n")
    return X_final, y, scaler

print("assemble_feature_matrix defined!")

assemble_feature_matrix defined!


In [32]:
#CALL the function
X, y, scaler = assemble_feature_matrix(df, tfidf_matrix)

STEP 7: Assembling final feature matrix...
 Final feature matrix shape : (17880, 3042)
 Target distribution        : [17014   866]



## STEP 9 : TRAIN TEST SPLIT before step 8

In [33]:
# define split function (step 9 )
def split_data(X, y, test_size = 0.2):
    print("STEP 9: Splitting data (80\20) stratified)...")
    X_train, X_test, y_train, y_test = train_test_split(
       X, y, test_size = test_size, random_state = 42, stratify = y
    )
    print(f" Train size : {X_train.shape[0]:,} samples")
    print(f" Test size  : {X_test.shape[0]:,} samples")
    return X_train, X_test, y_train, y_test

print("split_data defined!")

split_data defined!


## STEP 8 : HANDEL CLASS IMBALANCE WITH SMOTE before step 9

In [34]:
# Define SMOTE function (step 8)
def apply_smote(X_train, y_train):
    print("STEP 8: Applying SMOTE to balance classes...")
    print(f" Before SMOTE: {np.bincount(y_train)}")
    sm = SMOTE(random_state = 42, k_neighbors = 5)
    X_resampled, y_resampled = sm.fit_resample(X_train, y_train)
    print(f" After SMOTE : {np.bincount(y_resampled)}")
    return X_resampled, y_resampled

print("apply_smote defined!")

apply_smote defined!


In [35]:
# Call Step 9 First (SPlit before SMOTE)
X_train, X_test, y_train, y_test = split_data(X, y)

STEP 9: Splitting data (80) stratified)...
 Train size : 14,304 samples
 Test size  : 3,576 samples


In [36]:
# Call step 8 (SMOTE on train data only)
X_train_bal, y_train_bal = apply_smote(X_train, y_train)

STEP 8: Applying SMOTE to balance classes...
 Before SMOTE: [13611   693]
 After SMOTE : [13611 13611]


In [37]:
# Verify everything looks correct
print("Final shapes")
print(f" X_train balanced : {X_train_bal.shape}")
print(f" X_test           : {X_test.shape}")
print(f" y_train balanced : {np.bincount(y_train_bal)}")
print(f" y_test           : {np.bincount(y_test)}")

Final shapes
 X_train balanced : (27222, 3042)
 X_test           : (3576, 3042)
 y_train balanced : [13611 13611]
 y_test           : [3403  173]


## Step 10 : SAVE PROCESSED DATA

In [38]:
# define save function

In [39]:
def save_processed_data(df, X_train, X_test, y_train, y_test,
                        vectorizer, scaler, output_dir='processed_data'):
    import scipy.sparse as sp
    import pickle

    os.makedirs(output_dir, exist_ok=True)
    print(f" STEP 10: Saving artifacts to '{output_dir}/'...")

    sp.save_npz(f'{output_dir}/X_train.npz', X_train)
    sp.save_npz(f'{output_dir}/X_test.npz', X_test)
    np.save(f'{output_dir}/y_train.npy', y_train)
    np.save(f'{output_dir}/y_test.npy', y_test)

    with open(f'{output_dir}/tfidf_vectorizer.pkl', 'wb') as f:
        pickle.dump(vectorizer, f)
    with open(f'{output_dir}/scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)

    df.to_csv(f'{output_dir}/engineered_features.csv', index=False)

    print(f"  Saved: X_train.npz, X_test.npz, y_train.npy, y_test.npy")
    print(f"  Saved: tfidf_vectorizer.pkl, scaler.pkl")
    print(f"  Saved:  engineered_features.csv")

print("Save_processed_data defined!")

Save_processed_data defined!


## STEP 11 : EDA REPORT

In [40]:
# Define EDA report

In [41]:
def print_eda_report(df):
    print("=" * 60)
    print("QUICK EDA - FAKE vs REAL COMPARISON")
    print("=" * 60)

    fake = df[df['fraudulent'] == 1]
    real = df[df['fraudulent'] == 0]

    report_cols = [
        ('india_fraud_score',       'India fraud keyword score'),
        ('urgency_score',           'Urgency word count'),
        ('has_whatsapp_contact',    'Has WhatsApp contact (%)'),
        ('has_suspicious_email',    'Has suspicious email (%)'),
        ('has_fee_language',        'Has fee language (%)'),
        ('requests_personal_docs',  'Requests personal docs (%)'),
        ('has_portal_fraud_pattern','portal fraud pattern (%)'),
        ('has_linkedin_fraud',      'LinkedIn fraud pattern (%)'),
        ('company_profile_missing', 'Conpany profile missing (%)'),
        ('vague_job_title',         'Vague job title (%)'),
        ('sentiment_positive',      'Avg positive sentiment'),
        ('missing_fields_score',    'Missing fields score'),
        ('desc_word_count',         'Avg description word count'),
        ('exclamation_count',       'Avg exclamation marks'),
    ]

    print(f"  {'Metric' :<40} {'FAKE' :>10} {'REAL' :>10}")
    print(f"  {'-'*40} {'-'*10} {'-'*10}")
    for col, label in report_cols:
        if col in df.columns:
            fake_val = fake[col].mean()
            real_val = real[col].mean()
            print(f"  {label:<40} {fake_val:>10.3f} {real_val:>10.3f}")
    print()

print("print_eda_report defined!")

print_eda_report defined!


In [42]:
# call both function i.e: step 10 and 11

In [43]:
save_processed_data(df, X_train_bal, X_test, y_train_bal, y_test,
                    vectorizer, scaler)

print_eda_report(df)

 STEP 10: Saving artifacts to 'processed_data/'...
  Saved: X_train.npz, X_test.npz, y_train.npy, y_test.npy
  Saved: tfidf_vectorizer.pkl, scaler.pkl
  Saved:  engineered_features.csv
QUICK EDA - FAKE vs REAL COMPARISON
  Metric                                         FAKE       REAL
  ---------------------------------------- ---------- ----------
  India fraud keyword score                     0.013      0.003
  Urgency word count                            1.002      1.266
  Has WhatsApp contact (%)                      0.000      0.003
  Has suspicious email (%)                      0.003      0.047
  Has fee language (%)                          0.001      0.002
  Requests personal docs (%)                    0.001      0.005
  portal fraud pattern (%)                      0.003      0.019
  LinkedIn fraud pattern (%)                    0.000      0.000
  Conpany profile missing (%)                   0.678      0.160
  Vague job title (%)                           0.161      0.013

## VERIFYING ALL SAVED FILES EXIST

In [44]:
# Verify all saved files exist before starting model training
import os

files_needed = [
    'processed_data/X_train.npz',
    'processed_data/X_test.npz',
    'processed_data/y_train.npy',
    'processed_data/y_test.npy',
    'processed_data/tfidf_vectorizer.pkl',
    'processed_data/scaler.pkl',
    'processed_data/engineered_features.csv',
]

print("Checking saved files...")
all_good = True
for f in files_needed:
    if os.path.exists(f):
        print(f"  Found    : {f}")
    else:
        print(f"  MISSING  : {f}")
        all_good = False

if all_good:
    print("\nAll files found! Ready for model training!")
else:
    print("\nSome files missing — rerun save_processed_data() first!")

Checking saved files...
  Found    : processed_data/X_train.npz
  Found    : processed_data/X_test.npz
  Found    : processed_data/y_train.npy
  Found    : processed_data/y_test.npy
  Found    : processed_data/tfidf_vectorizer.pkl
  Found    : processed_data/scaler.pkl
  Found    : processed_data/engineered_features.csv

All files found! Ready for model training!


# Model Training

In [45]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

In [46]:
#!pip install shap

In [47]:
# ML Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                            f1_score, precision_score, recall_score,roc_auc_score, roc_curve)
import xgboost as xgb
import shap

In [48]:
# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, LSTM, Dense, Dropout,
                                      Bidirectional, GlobalMaxPooling1D)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

In [49]:
# Sparse matrix Loading
import scipy.sparse as sp

os.makedirs('models', exist_ok = True)
os.makedirs('plots', exist_ok = True)

## STEP 1: LOAD SAVED PROCESSED DATA

In [50]:
def load_processed_data(data_dir = 'processed_data'):
    print("=" * 60)
    print("LOADING PREPROCESSED DATA...")
    print("=" * 60)

    X_train = sp.load_npz(f'{data_dir}/X_train.npz')
    X_test  = sp.load_npz(f'{data_dir}/X_test.npz')
    y_train = np.load(f'{data_dir}/y_train.npy')
    y_test  = np.load(f'{data_dir}/y_test.npy')

    with open(f'{data_dir}/tfidf_vectorizer.pkl', 'rb') as f:
        vectorizer = pickle.load(f)
    with open(f'{data_dir}/scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)

    df= pd.read_csv(f'{data_dir}/engineered_features.csv')

    print(f"  X_train shape : {X_train.shape}")
    print(f"  X_test shape  : {X_test.shape}")
    print(f"  y_train dist  : {np.bincount(y_train)}")
    print(f"  y_test dist   : {np.bincount(y_test)}")
    print()
    return X_train, X_test, y_train, y_test, vectorizer, scaler, df

print("load_processed_data defined!")

load_processed_data defined!


In [51]:
X_train, X_test, y_train, y_test, vectorizer, scaler, df = load_processed_data()

LOADING PREPROCESSED DATA...
  X_train shape : (27222, 3042)
  X_test shape  : (3576, 3042)
  y_train dist  : [13611 13611]
  y_test dist   : [3403  173]



## STEP 2: EVALUATION HELPER

In [52]:
def evaluate_model(model_name, y_test, y_pred, y_prob = None):
    print(f"\n{'=' * 60}")
    print(f" RESULTS - {model_name}")
    print(f"{'=' * 60}")

    print(classification_report(y_test, y_pred, target_names = ['Real Job', 'Fake Job']))

    f1         = f1_score(y_test, y_pred)
    precision  = precision_score(y_test, y_pred)
    recall     = recall_score(y_test, y_pred)

    print(f"  F1 Score  :{f1:.4f}")
    print(f"  Precision :{precision:.4f}")
    print(f"  Recall    :{recall:.4f}")

    if y_prob is not None:
        auc = roc_auc_score(y_test, y_prob)
        print(f"  ROC_AUC   : {auc:.4f}")
    else:
        auc = None

    #Confusion matrix plot
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize = (6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap = 'Blues',
                xticklabels = ['Real', 'Fake'],
                yticklabels = ['Real', 'Fake'])
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'plots/confusion_matrix_{model_name.replace(" ", "_")}.png',
                dpi=150)
    plt.close()
    print(f" Confusion matrix saved to plots/")

    return {'model' : model_name, 'f1' : f1,
            'precision' : precision, 'recall' : recall, 'auc' : auc}
    
print("evaluate_model defined!")

evaluate_model defined!


In [53]:
X_train, X_test, y_train, y_test, vectorizer, scaler, df = load_processed_data()

LOADING PREPROCESSED DATA...
  X_train shape : (27222, 3042)
  X_test shape  : (3576, 3042)
  y_train dist  : [13611 13611]
  y_test dist   : [3403  173]



## STEP 3 : MODEL 1 - RANDOM FOREST (BASELINE) 

In [54]:
def train_random_forest(X_train, X_test, y_train, y_test):
    print("\n" + "=" * 60)
    print(" MODEL 1 - RANDOM FOREST (BASELINE)")
    print("=" * 60)
    print( "Training.... (3-5 minutes)")

    rf = RandomForestClassifier(
         n_estimators  = 200,
         max_depth    = 20,
         min_samples_split = 5,
         min_samples_leaf  = 2,
         class_weight      = 'balanced',
         random_state      = 42,
         n_jobs            = -1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)
    y_prob = rf.predict_proba(X_test)[:, 1]

    results = evaluate_model('Random Forest', y_test, y_pred, y_prob)

    # Save model
    with open('models/random_forest.pkl', 'wb') as f:
        pickle.dump(rf, f)
    print(" Saved: models/random_forest.pkl")

    return rf, results

print("train_random_forest defined!")

train_random_forest defined!


In [55]:
rf_model, rf_results = train_random_forest(X_train, X_test, y_train, y_test)


 MODEL 1 - RANDOM FOREST (BASELINE)
Training.... (3-5 minutes)

 RESULTS - Random Forest
              precision    recall  f1-score   support

    Real Job       0.99      0.98      0.98      3403
    Fake Job       0.60      0.74      0.66       173

    accuracy                           0.96      3576
   macro avg       0.80      0.86      0.82      3576
weighted avg       0.97      0.96      0.97      3576

  F1 Score  :0.6649
  Precision :0.6038
  Recall    :0.7399
  ROC_AUC   : 0.9783
 Confusion matrix saved to plots/
 Saved: models/random_forest.pkl


## STEP 4: MODEL 2 - LSTM (ON TEXT ONLY)

In [56]:
def train_lstm(df, y_train_full, y_test_full,
               max_words = 10000, max_len = 200, epochs = 10):
    print("\n" + "=" * 60)
    print(" MODEL 2 - LSTM (TEXT SEQUENCE MODEL)")
    print("=" * 60)
    print( " Preparing text sequences...")

    #Use cleaned description text
    texts = df['cleaned_description'].fillna('').values

    #Tokenize
    tokenizer = Tokenizer(num_words = max_words, oov_token = '<OOV>')
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    padded    = pad_sequences(sequences, maxlen = max_len, padding = 'post',
                                truncating = 'post')

    # Recreate same 80/20 split indices
    from sklearn.model_selection import train_test_split
    y_all = df['fraudulent'].values
    idx   = np.arange(len(df))
    idx_train, idx_test = train_test_split(
        idx, test_size = 0.2, random_state = 42, stratify = y_all
    )

    X_text_train = padded[idx_train]
    X_text_test  = padded[idx_test]
    y_tr         = y_all[idx_train]
    y_te         = y_all[idx_test]


    print(f" Vocabulary size : {len(tokenizer.word_index)}")
    print(f" Max sequence len: {max_len}")
    print(" Building LSTM model...")

    # Build model
    model = Sequential([
        Embedding(max_words, 64, input_length = max_len),
        Bidirectional(LSTM(64, return_sequences = True)),
        GlobalMaxPooling1D(),
        Dense(64, activation = 'relu'),
        Dropout(0.4),
        Dense(32, activation = 'relu'),
        Dropout(0.3),
        Dense(1, activation = 'sigmoid')
    ])
    
    model.compile(
        optimizer = 'adam',
        loss = 'binary_crossentropy',
        metrics = ['accuracy']
     )

    model.summary()

    #Early stopping to prevent overfitting
    early_stop = EarlyStopping(
        monitor  = 'val_loss',
        patience = 3,
        restore_best_weights = True
    )

    print("\n  Training LSTM...(10-15 minutes)")
    history = model.fit(
        X_text_train, y_tr,
        epochs          = epochs,
        batch_size      = 64,
        validation_split= 0.1,
        callbacks       = [early_stop],
        verbose         = 1
    )

    # Evaluate
    y_prob = model.predict(X_text_test).flatten()
    y_pred = (y_prob >= 0.5).astype(int)

    results = evaluate_model('LSTM', y_te, y_pred, y_prob)

    # Plot taining history
    plt.figure(figsize = (10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'],    label= 'Train Loss')
    plt.plot(history.history['val_loss'],label= 'Val Loss')
    plt.title('LSTM Loss')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'],    label = 'Train Acc')
    plt.plot(history.history['val_accuracy'],label = 'Val Acc')
    plt.title('LSTM Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.savefig('plots/lstm_training_history.png', dpi = 150)
    plt.close()
    print(" Training history plot saved to plots/")

    # Save model and tokenizer
    model.save('models/lstm_model.h5')
    with open('models/lstm_tokenizer.pkl', 'wb') as f:
        pickle.dump(tokenizer, f)
    print(" Saved: models/lstm_model.h5")
    print(" Saved: models/lstm_tokenizer.pkl")

    return model, tokenizer, results

print("train_lstm defined!")

train_lstm defined!


In [57]:
lstm_model, lstm_tokenizer, lstm_results = train_lstm(df, y_train, y_test)


 MODEL 2 - LSTM (TEXT SEQUENCE MODEL)
 Preparing text sequences...
 Vocabulary size : 54257
 Max sequence len: 200
 Building LSTM model...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ ?                           │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


  Training LSTM...(10-15 minutes)
Epoch 1/10
202/202 ━━━━━━━━━━━━━━━━━━━━ 57s 244ms/step - accuracy: 0.9277 - loss: 0.2958 - val_accuracy: 0.9602 - val_loss: 0.1205
Epoch 2/10
202/202 ━━━━━━━━━━━━━━━━━━━━ 49s 245ms/step - accuracy: 0.9617 - loss: 0.1411 - val_accuracy: 0.9693 - val_loss: 0.1159
Epoch 3/10
202/202 ━━━━━━━━━━━━━━━━━━━━ 47s 232ms/step - accuracy: 0.9705 - loss: 0.0863 - val_accuracy: 0.9734 - val_loss: 0.0880
Epoch 4/10
202/202 ━━━━━━━━━━━━━━━━━━━━ 48s 238ms/step - accuracy: 0.9881 - loss: 0.0450 - val_accuracy: 0.9783 - val_loss: 0.0982
Epoch 5/10
202/202 ━━━━━━━━━━━━━━━━━━━━ 47s 232ms/step - accuracy: 0.9931 - loss: 0.0259 - val_accuracy: 0.9748 - val_loss: 0.1253
Epoch 6/10
202/202 ━━━━━━━━━━━━━━━━━━━━ 54s 265ms/step - accuracy: 0.9938 - loss: 0.0193 - val_accuracy: 0.9748 - val_loss: 0.1067
112/112 ━━━━━━━━━━━━━━━━━━━━ 7s 61ms/step

 RESULTS - LSTM
              precision    recall  f1-score   support

    Real Job       0.98      0.99      0.99      3403
    Fake Jo

 Training history plot saved to plots/
 Saved: models/lstm_model.h5
 Saved: models/lstm_tokenizer.pkl


## STEP 5: MODEL 3 - XGBOOST (FINAL MODEL)

In [60]:
def train_xgboost(X_train, X_test, y_train, y_test):
    print("\n" + "="*60)
    print("  MODEL 3 — XGBOOST")
    print("="*60)
    print("  Training XGBoost... (5-8 minutes)")

    neg   = np.sum(y_train == 0)
    pos   = np.sum(y_train == 1)
    scale = round(neg / pos, 2)
    print(f"  scale_pos_weight: {scale}")

    xgb_model = xgb.XGBClassifier(
        n_estimators          = 500,
        max_depth             = 6,
        learning_rate         = 0.05,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        min_child_weight      = 5,
        gamma                 = 0.1,
        reg_alpha             = 0.1,
        reg_lambda            = 1.0,
        scale_pos_weight      = scale,
        early_stopping_rounds = 30,
        eval_metric           = 'logloss',
        random_state          = 42,
        n_jobs                = -1,
        verbosity             = 1
    )

    X_tr_dense = X_train.toarray()
    X_te_dense = X_test.toarray()
    eval_set   = [(X_tr_dense, y_train), (X_te_dense, y_test)]

    xgb_model.fit(
        X_tr_dense, y_train,
        eval_set = eval_set,
        verbose  = 50
    )

    y_pred  = xgb_model.predict(X_te_dense)
    y_prob  = xgb_model.predict_proba(X_te_dense)[:, 1]
    results = evaluate_model('XGBoost', y_test, y_pred, y_prob)

    plt.figure(figsize=(8, 4))
    evals = xgb_model.evals_result()
    plt.plot(evals['validation_0']['logloss'], label='Train')
    plt.plot(evals['validation_1']['logloss'], label='Val')
    plt.title('XGBoost Learning Curve')
    plt.xlabel('Rounds')
    plt.ylabel('LogLoss')
    plt.legend()
    plt.tight_layout()
    plt.savefig('plots/xgboost_learning_curve.png', dpi=150)
    plt.close()
    print("  Learning curve saved!")

    # Save model - pickle only (JSON save has a bug in XGBoost 2.x)
    with open('models/xgboost_model.pkl', 'wb') as f:
        pickle.dump(xgb_model, f)
    # xgb_model.save_model('models/xgboost_model.json')  # ← remove this line
    print("  Saved: models/xgboost_model.pkl")
    
    return xgb_model, X_te_dense, results

print("train_xgboost fixed!")

train_xgboost fixed!


In [61]:
xgb_model, X_test_dense, xgb_results = train_xgboost(X_train, X_test, y_train, y_test)


  MODEL 3 — XGBOOST
  Training XGBoost... (5-8 minutes)
  scale_pos_weight: 1.0
[0]	validation_0-logloss:0.65567	validation_1-logloss:0.65691
[50]	validation_0-logloss:0.12885	validation_1-logloss:0.15565
[100]	validation_0-logloss:0.07004	validation_1-logloss:0.09987
[150]	validation_0-logloss:0.04880	validation_1-logloss:0.08120
[200]	validation_0-logloss:0.03719	validation_1-logloss:0.07086
[250]	validation_0-logloss:0.03077	validation_1-logloss:0.06516
[300]	validation_0-logloss:0.02483	validation_1-logloss:0.06092
[350]	validation_0-logloss:0.02042	validation_1-logloss:0.05751
[400]	validation_0-logloss:0.01680	validation_1-logloss:0.05500
[450]	validation_0-logloss:0.01435	validation_1-logloss:0.05282
[499]	validation_0-logloss:0.01248	validation_1-logloss:0.05185

 RESULTS - XGBoost
              precision    recall  f1-score   support

    Real Job       0.99      1.00      0.99      3403
    Fake Job       0.89      0.76      0.82       173

    accuracy                      

## define feature importance function:

In [64]:
def run_feature_importance(xgb_model, vectorizer):
    print("\n" + "="*60)
    print("  STEP 6 — FEATURE IMPORTANCE")
    print("="*60)

    tfidf_names      = vectorizer.get_feature_names_out().tolist()
    engineered_names = [
        'desc_word_count', 'desc_char_count', 'desc_sentence_count',
        'title_word_count', 'avg_word_length',
        'exclamation_count', 'question_mark_count', 'caps_ratio',
        'india_fraud_score', 'urgency_score', 'overpromise_score',
        'has_whatsapp_contact', 'has_suspicious_email', 'has_phone_in_desc',
        'has_fee_language', 'requests_personal_docs',
        'has_linkedin_fraud', 'has_portal_fraud_pattern',
        'sentiment_compound', 'sentiment_positive',
        'sentiment_negative', 'sentiment_neutral', 'repetition_score',
        'salary_min', 'salary_max', 'salary_missing', 'salary_range_width',
        'salary_too_high_for_fresher', 'salary_unrealistic',
        'company_profile_missing', 'company_profile_length',
        'has_company_logo', 'has_questions', 'vague_job_title',
        'requirements_missing', 'requirements_length', 'benefits_missing',
        'employment_type_encoded', 'experience_encoded',
        'generic_location', 'missing_fields_score', 'telecommuting'
    ]
    all_names  = tfidf_names + engineered_names
    importance = xgb_model.feature_importances_
    n          = min(len(importance), len(all_names))
    importance = importance[:n]
    all_names  = all_names[:n]

    indices    = np.argsort(importance)[::-1][:20]
    top_feats  = [all_names[i] for i in indices]
    top_scores = [importance[i] for i in indices]

    plt.figure(figsize=(10, 8))
    plt.barh(range(20), top_scores[::-1], color='steelblue')
    plt.yticks(range(20), top_feats[::-1])
    plt.xlabel('Importance Score')
    plt.title('Top 20 Features — XGBoost')
    plt.tight_layout()
    plt.savefig('plots/feature_importance.png', dpi=150)
    plt.close()
    print("  Saved: plots/feature_importance.png")

    print(f"\n  {'Rank':<6} {'Feature':<35} {'Score':>8}")
    print(f"  {'-'*6} {'-'*35} {'-'*8}")
    for rank, (feat, score) in enumerate(
            zip(top_feats[:10], top_scores[:10]), 1):
        print(f"  {rank:<6} {feat:<35} {score:>8.4f}")

    with open('models/feature_names.pkl', 'wb') as f:
        pickle.dump(all_names, f)
    print("\n  Saved: models/feature_names.pkl")
    return top_feats, top_scores

print("run_feature_importance defined!")

run_feature_importance defined!


In [65]:
top_features, top_scores = run_feature_importance(xgb_model, vectorizer)


  STEP 6 — FEATURE IMPORTANCE
  Saved: plots/feature_importance.png

  Rank   Feature                                Score
  ------ ----------------------------------- --------
  1      encouraged                            0.0606
  2      company_profile_missing               0.0340
  3      work home                             0.0213
  4      applicable                            0.0186
  5      financing                             0.0159
  6      owner                                 0.0152
  7      designed                              0.0140
  8      company_profile_length                0.0132
  9      corporate                             0.0127
  10     has_company_logo                      0.0120

  Saved: models/feature_names.pkl


## define compare function

In [66]:
def compare_models(results_list):
    print("\n" + "="*60)
    print("  MODEL COMPARISON")
    print("="*60)
    df_res = pd.DataFrame(results_list).sort_values('f1', ascending=False)
    print(f"\n  {'Model':<20} {'F1':>8} {'Precision':>10} "
          f"{'Recall':>8} {'AUC':>8}")
    print(f"  {'-'*20} {'-'*8} {'-'*10} {'-'*8} {'-'*8}")
    for _, row in df_res.iterrows():
        auc_str = f"{row['auc']:.4f}" if row['auc'] else "  N/A  "
        print(f"  {row['model']:<20} {row['f1']:>8.4f} "
              f"{row['precision']:>10.4f} {row['recall']:>8.4f} "
              f"{auc_str:>8}")

    metrics = ['f1', 'precision', 'recall']
    x       = np.arange(len(df_res))
    width   = 0.25
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, metric in enumerate(metrics):
        ax.bar(x + i*width, df_res[metric], width, label=metric.capitalize())
    ax.set_xticks(x + width)
    ax.set_xticklabels(df_res['model'])
    ax.set_ylabel('Score')
    ax.set_title('Model Comparison')
    ax.legend()
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('plots/model_comparison.png', dpi=150)
    plt.close()
    print("\n  Saved: plots/model_comparison.png")
    best = df_res.iloc[0]
    print(f"\n  Best model : {best['model']}")
    print(f"  F1 Score   : {best['f1']:.4f}")
    return df_res

print("compare_models defined!")

compare_models defined!


In [67]:
df_results = compare_models([rf_results, lstm_results, xgb_results])


  MODEL COMPARISON

  Model                      F1  Precision   Recall      AUC
  -------------------- -------- ---------- -------- --------
  XGBoost                0.8199     0.8859   0.7630   0.9863
  Random Forest          0.6649     0.6038   0.7399   0.9783
  LSTM                   0.6386     0.8125   0.5260   0.9549

  Saved: plots/model_comparison.png

  Best model : XGBoost
  F1 Score   : 0.8199


## define save summary function

In [68]:
def save_model_summary(df_results, vectorizer, scaler):
    import json
    best    = df_results.iloc[0]
    summary = {
        'best_model'  : best['model'],
        'f1_score'    : round(float(best['f1']), 4),
        'precision'   : round(float(best['precision']), 4),
        'recall'      : round(float(best['recall']), 4),
        'auc'         : round(float(best['auc']), 4) if best['auc'] else None,
        'model_file'  : 'models/xgboost_model.pkl',
        'version'     : '2.0',
        'last_updated': 'March 2026',
        'author'      : 'Divya Virkud'
    }
    with open('models/model_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    print("\n  Saved: models/model_summary.json")
    for k, v in summary.items():
        print(f"    {k:<20}: {v}")

print("save_model_summary defined!")

save_model_summary defined!


## CALL save summary

In [69]:
save_model_summary(df_results, vectorizer, scaler)


  Saved: models/model_summary.json
    best_model          : XGBoost
    f1_score            : 0.8199
    precision           : 0.8859
    recall              : 0.763
    auc                 : 0.9863
    model_file          : models/xgboost_model.pkl
    version             : 2.0
    last_updated        : March 2026
    author              : Divya Virkud


## final check

In [70]:
print("\n" + "="*60)
print("  MODEL TRAINING COMPLETE!")
print("="*60)

saved_files = [
    'models/random_forest.pkl',
    'models/lstm_model.h5',
    'models/xgboost_model.pkl',
    'models/xgboost_model.json',
    'models/feature_names.pkl',
    'models/model_summary.json',
    'plots/confusion_matrix_Random_Forest.png',
    'plots/confusion_matrix_LSTM.png',
    'plots/confusion_matrix_XGBoost.png',
    'plots/feature_importance.png',
    'plots/model_comparison.png',
    'plots/xgboost_learning_curve.png',
]
print("\nChecking saved files:")
for f in saved_files:
    status = "Found" if os.path.exists(f) else "MISSING"
    print(f"  {status:<8} : {f}")

print("\nNext step: Build Streamlit app!")


  MODEL TRAINING COMPLETE!

Checking saved files:
  Found    : models/random_forest.pkl
  Found    : models/lstm_model.h5
  Found    : models/xgboost_model.pkl
  MISSING  : models/xgboost_model.json
  Found    : models/feature_names.pkl
  Found    : models/model_summary.json
  Found    : plots/confusion_matrix_Random_Forest.png
  Found    : plots/confusion_matrix_LSTM.png
  Found    : plots/confusion_matrix_XGBoost.png
  Found    : plots/feature_importance.png
  Found    : plots/model_comparison.png
  Found    : plots/xgboost_learning_curve.png

Next step: Build Streamlit app!
